# Frequency calibration workflow (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

This notebook focuses on the two frequency-calibration loops you will use
most often after initial bring-up: qubit control frequency and readout
frequency. Adjust the setup cell for your system before you run it.

## 1. Create an `Experiment`

Start with one qubit whose readout and drive channels are already wired
and visible in the configuration.

In [1]:
import numpy as np

import qubex_emulator as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect and collect a baseline

A waveform check plus one Rabi measurement gives you a good baseline
before you change the frequencies.

In [2]:
exp.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-cr-cr', drag_hpi_puls

In [3]:
waveform_result = exp.check_waveform()
rabi_result = exp.obtain_rabi_params()

## 3. Run a coarse control-frequency sweep

Use a broad chevron scan first so you can see the control-frequency region before the narrower calibration step.

In [4]:
# Obtain the Chevron pattern
chevron_result = exp.chevron_pattern(
    exp.qubit_labels,
    detuning_range=np.linspace(-0.05, 0.05, 51),
    time_range=np.arange(0, 201, 4),
)

## 4. Calibrate the control frequency

Sweep a detuning window around the current qubit frequency and let Qubex
fit the updated resonance.

In [5]:
control_frequency_result = exp.calibrate_control_frequency(
    targets=exp.qubit_labels,
    detuning_range=np.linspace(-0.01, 0.01, 21),
    time_range=np.arange(0, 101, 4),
)

In [6]:
# Update `control_frequency.yaml` manually, then reload
exp.reload()

## 5. Re-check the Rabi oscillation

Running `obtain_rabi_params()` again quickly shows whether the updated
qubit frequency moved the oscillation closer to the expected model.

In [7]:
updated_rabi_result = exp.obtain_rabi_params()

## 6. Calibrate the readout frequency

Use a narrower detuning window once the qubit drive side is in place.

In [8]:
readout_frequency_result = exp.calibrate_readout_frequency(
    targets=exp.qubit_labels,
    detuning_range=np.linspace(-0.01, 0.01, 21),
    time_range=np.arange(0, 101, 4),
)

In [9]:
# Update `readout_frequency.yaml` manually, then reload
exp.reload()

## 7. Save the calibration note

Save the updated calibration note so later notebooks start from the newly
fitted frequencies.

In [10]:
exp.calib_note.save()